In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import re
import html

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import save_npz
from sentence_transformers import SentenceTransformer

c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_path = RAW_DATA_DIR / "banking77_train.csv"

train_df = pd.read_csv(train_path)

print("Training shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())
print("Unique intents:", train_df["category"].nunique())

Training shape: (10003, 2)
Columns: ['text', 'category']
Unique intents: 77


In [4]:
def normalize_text(text):
    text = html.unescape(str(text))
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [9]:

X = train_df["text"].apply(normalize_text)
y = train_df["category"]

print("Number of messages:", len(X))
print("Number of intent classes:", y.nunique())

display(pd.DataFrame({
    "text": train_df["text"].head(5),
    "normalized_text": X.head(5),
    "category": y.head(5)}))

Number of messages: 10003
Number of intent classes: 77


,text,normalized_text,category
0,I am still waiting on my card?,i am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,i have been waiting over a week. is the card s...,card_arrival
3,Can I track my card while it is in the process...,can i track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...","how do i know if i will get my card, or if it ...",card_arrival


### training and validation split

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Training labels:", y_train.nunique())
print("Validation labels:", y_val.nunique())

Training samples: 8002
Validation samples: 2001
Training labels: 77
Validation labels: 77


### Bag of Words

In [11]:
bow_vectorizer = CountVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    max_features=20000
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)

print("BoW training shape:", X_train_bow.shape)
print("BoW validation shape:", X_val_bow.shape)
print("Vocabulary size:", len(bow_vectorizer.vocabulary_))

BoW training shape: (8002, 20000)
BoW validation shape: (2001, 20000)
Vocabulary size: 20000


### Inspect the BoW representation

In [12]:
bow_features = bow_vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(
    X_train_bow[:5].toarray(),
    columns=bow_features)

display(bow_df.iloc[:, :20])

,00,00 in,000,000 but,000 from,000 to,10,10 000,10 atm,10 minutes,10 pounds,10 still,10 the,10 transactions,10 why,100,100 all,100 but,18,18 to
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### TF-IDF

In [13]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF training shape: (8002, 20000)
TF-IDF validation shape: (2001, 20000)
Vocabulary size: 20000


### Compare BoW and TF-IDF features

In [14]:
print("BoW sample feature values:")
print(X_train_bow[0, :20].toarray())

print("\nTF-IDF sample feature values:")
print(X_train_tfidf[0, :20].toarray())

BoW sample feature values:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

TF-IDF sample feature values:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


### Save vectorizers and sparse matrices

In [15]:
save_npz(PROCESSED_DATA_DIR / "X_train_bow.npz",X_train_bow)

save_npz(PROCESSED_DATA_DIR / "X_val_bow.npz",X_val_bow)

save_npz(PROCESSED_DATA_DIR / "X_train_tfidf.npz",X_train_tfidf)

save_npz(PROCESSED_DATA_DIR / "X_val_tfidf.npz",X_val_tfidf)

joblib.dump(bow_vectorizer,
    MODELS_DIR / "bow_vectorizer.joblib")

joblib.dump(tfidf_vectorizer,
    MODELS_DIR / "tfidf_vectorizer.joblib")

pd.DataFrame({
    "text": X_train,
    "category": y_train}).to_csv(
    PROCESSED_DATA_DIR / "train_split.csv",
    index=False)

pd.DataFrame({
    "text": X_val,
    "category": y_val}).to_csv(
    PROCESSED_DATA_DIR / "validation_split.csv",
    index=False)

print("BoW and TF-IDF artifacts saved successfully.")

BoW and TF-IDF artifacts saved successfully.


### Sentence embeddings

In [16]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True)

X_val_embeddings = embedding_model.encode(
    X_val.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True)

print("Training embeddings shape:", X_train_embeddings.shape)
print("Validation embeddings shape:", X_val_embeddings.shape)

c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RAHUL MANGUTKAR\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\RAHUL MANGUTKAR\banking-customer-intelligence\.ven

Training embeddings shape: (8002, 384)
Validation embeddings shape: (2001, 384)


### Save sentence embeddings

In [17]:
np.save(
    PROCESSED_DATA_DIR / "X_train_embeddings.npy",
    X_train_embeddings)

np.save(
    PROCESSED_DATA_DIR / "X_val_embeddings.npy",
    X_val_embeddings)

print("Sentence embeddings saved successfully.")

Sentence embeddings saved successfully.


In [18]:
y_train.to_csv(
    PROCESSED_DATA_DIR / "y_train.csv",
    index=False,
    header=["category"])

y_val.to_csv(
    PROCESSED_DATA_DIR / "y_val.csv",
    index=False,
    header=["category"])

print("Training and validation labels saved.")

Training and validation labels saved.


### Final validation checks

In [19]:
assert X_train_bow.shape[0] == len(y_train)
assert X_val_bow.shape[0] == len(y_val)

assert X_train_tfidf.shape[0] == len(y_train)
assert X_val_tfidf.shape[0] == len(y_val)

assert X_train_bow.shape[1] == X_val_bow.shape[1]
assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1]

print("All BoW and TF-IDF validation checks passed.")

All BoW and TF-IDF validation checks passed.


In [20]:
assert X_train_embeddings.shape[0] == len(y_train)
assert X_val_embeddings.shape[0] == len(y_val)

print("Embedding validation checks passed.")

Embedding validation checks passed.


### observations

Bag of Words represents messages through word and phrase counts.

TF-IDF assigns weights based on term frequency and how widely terms occur across documents.

Sentence embeddings provide dense vector representations that can capture semantic similarities.

The BoW and TF-IDF vocabularies were fitted on the training split only and then applied to validation data.

The official test dataset was kept separate from training and validation.

Sparse matrices were used for BoW and TF-IDF to reduce memory usage.

Sentence embeddings require additional model resources and computation.